# 02 · Data Preparation

Building the final table: one row per team per season. Decisions and reasons go in `docs/project_notes.md`.

In [1]:
import sys
from pathlib import Path
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loading import load_all

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)   # plots for the slides are saved here

tables = load_all()

## Team table (task a)

In [2]:
# Which data we use. Change these two lines if we decide otherwise.
LEAGUES = ["NHL"]
FIRST_YEAR = 1979

teams = tables["Teams"]
nhl = teams[teams["lgID"].isin(LEAGUES)].copy()

# Results per game, because seasons had different lengths
nhl["pts_per_game"] = nhl["Pts"] / nhl["G"]
nhl["gf_per_game"] = nhl["GF"] / nhl["G"]
nhl["ga_per_game"] = nhl["GA"] / nhl["G"]

# Goals compared to that season's league average (1.0 = average), because scoring levels changed over the years
nhl["gf_vs_avg"] = nhl["gf_per_game"] / nhl.groupby("year")["gf_per_game"].transform("mean")
nhl["ga_vs_avg"] = nhl["ga_per_game"] / nhl.groupby("year")["ga_per_game"].transform("mean")

nhl["made_playoffs"] = nhl["playoff"].notna().astype(int)

# How many teams were in the same division that season
nhl["div_size"] = nhl.groupby(["year", "divID"])["tmID"].transform("count")

# Last season's results for the same club (franchID follows a club when it moves city)
nhl = nhl.sort_values(["franchID", "year"])
for col in ["pts_per_game", "gf_vs_avg", "ga_vs_avg", "made_playoffs", "rank"]:
    nhl["last_" + col] = nhl.groupby("franchID")[col].shift(1)
nhl["new_team"] = nhl["last_rank"].isna().astype(int)

team_table = nhl[nhl["year"] >= FIRST_YEAR][[
    "year", "franchID", "tmID", "name", "divID", "div_size",
    "rank",
    "last_pts_per_game", "last_gf_vs_avg", "last_ga_vs_avg",
    "last_made_playoffs", "last_rank", "new_team",
]].reset_index(drop=True)

print(team_table.shape)
team_table.head()

(753, 13)


,year,franchID,tmID,name,divID,div_size,rank,last_pts_per_game,last_gf_vs_avg,last_ga_vs_avg,last_made_playoffs,last_rank,new_team
0,1993,ANA,ANA,Mighty Ducks of Anaheim,PC,6.0,4,NaN,NaN,NaN,NaN,NaN,1
1,1994,ANA,ANA,Mighty Ducks of Anaheim,PC,6.0,6,0.845238,0.840842,0.921621,0.0,4.0,0
2,1995,ANA,ANA,Mighty Ducks of Anaheim,PC,7.0,4,0.770833,0.872015,1.144084,0.0,6.0,0
3,1996,ANA,ANA,Mighty Ducks of Anaheim,PC,7.0,2,0.951220,0.907924,0.958364,0.0,4.0,0
4,1997,ANA,ANA,Mighty Ducks of Anaheim,PC,7.0,6,1.036585,1.024775,0.974582,1.0,2.0,0


In [3]:
Path("../data").mkdir(exist_ok=True)
team_table.to_csv("../data/team_table.csv", index=False)

## Coach information (task b)

## Data quality, missing values and outliers